In [ ]:
!pip install lpips
!pip install torchmetrics


   ---------------------------------------- 0.0/983.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/983.0 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/983.0 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/983.0 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/983.0 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/983.0 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/983.0 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/983.0 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/983.0 kB ? eta -:--:--
   -------------------- ----------------- 524.3/983.0 kB 190.7 kB/s eta 0:00:03
   -------------------- ----------------- 524.3/983.0 kB 190.7 kB/s eta 0:00:03
   ------------------------------ ------- 786.4/983.0 kB 291.9 kB/s eta 0:00:01
   ---------------------------------------- 983.0/983.0 kB 357.7 kB/s  0:00:02

   ------------------------

In [2]:
import os
import torch
import lpips
import numpy as np
from PIL import Image
from torchvision import transforms
from torchmetrics.image import MultiScaleStructuralSimilarityIndexMeasure


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((64, 64)),   # adjust to your GAN image size
    transforms.ToTensor()
])


In [7]:
# MS-SSIM
from torchmetrics.image import MultiScaleStructuralSimilarityIndexMeasure

# Use only 3 scales instead of 5 (works with 64×64 images)
ms_ssim_metric = MultiScaleStructuralSimilarityIndexMeasure(data_range=1.0, betas=(0.0448, 0.2856, 0.3001)).to(device)


# LPIPS
lpips_model = lpips.LPIPS(net='alex').to(device)   # or net='vgg'


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: c:\Users\devgo\OneDrive\Desktop\fid\.venv\Lib\site-packages\lpips\weights\v0.1\alex.pth


In [8]:
def compute_ms_ssim_lpips(real_dir, fake_dir, max_images=100):
    real_files = sorted([os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.endswith(('png','jpg','jpeg')) and f.startswith("ISIC")])
    fake_files = sorted([os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.endswith(('png','jpg','jpeg')) and not f.startswith("ISIC")])
    
    n = min(len(real_files), len(fake_files), max_images)
    ms_ssim_scores, lpips_scores = [], []
    
    for i in range(n):
        real_img = transform(Image.open(real_files[i]).convert("RGB")).unsqueeze(0).to(device)
        fake_img = transform(Image.open(fake_files[i]).convert("RGB")).unsqueeze(0).to(device)
        
        # MS-SSIM
        ms_ssim_val = ms_ssim_metric(real_img, fake_img).item()
        ms_ssim_scores.append(ms_ssim_val)
        
        # LPIPS
        lpips_val = lpips_model(real_img, fake_img).item()
        lpips_scores.append(lpips_val)
    
    return np.mean(ms_ssim_scores), np.mean(lpips_scores)


In [9]:
root_dir = "C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data\\processed_images"

all_ms_ssim, all_lpips = [], []

for cls in os.listdir(root_dir):
    class_path = os.path.join(root_dir, cls)
    if os.path.isdir(class_path):
        ms, lp = compute_ms_ssim_lpips(class_path, class_path, max_images=100)
        print(f"Class: {cls} | MS-SSIM: {ms:.4f} | LPIPS: {lp:.4f}")
        all_ms_ssim.append(ms)
        all_lpips.append(lp)

print("\nOverall MS-SSIM:", np.mean(all_ms_ssim))
print("Overall LPIPS:", np.mean(all_lpips))


Class: AKIEC | MS-SSIM: 0.4283 | LPIPS: 0.2091
Class: BCC | MS-SSIM: 0.4474 | LPIPS: 0.2458
Class: BKL | MS-SSIM: 0.4148 | LPIPS: 0.2622
Class: DF | MS-SSIM: 0.4647 | LPIPS: 0.2212
Class: MEL | MS-SSIM: 0.4003 | LPIPS: 0.2619
Class: NV | MS-SSIM: nan | LPIPS: nan
Class: real_images_resized_128 | MS-SSIM: nan | LPIPS: nan


c:\Users\devgo\OneDrive\Desktop\fid\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\devgo\OneDrive\Desktop\fid\.venv\Lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Class: VASC | MS-SSIM: 0.5043 | LPIPS: 0.2559

Overall MS-SSIM: nan
Overall LPIPS: nan


In [10]:
import os
import random
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
from torchvision import transforms

from torchmetrics.image import MultiScaleStructuralSimilarityIndexMeasure
import lpips


In [11]:
# Path to your processed_images root (has subfolders per class)
root_dir = r"C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data\\processed_images"

# For reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# We use 128×128 to support small images and keep speed reasonable
IMG_SIZE = 128

# Transforms
to_01 = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),                 # -> [0,1]
])

# LPIPS expects [-1,1]
def to_m11(t):
    return t * 2.0 - 1.0


Using device: cuda


In [12]:
# MS-SSIM with 3 scales (works for small images)
ms_ssim_metric = MultiScaleStructuralSimilarityIndexMeasure(
    data_range=1.0,
    betas=(0.0448, 0.2856, 0.3001)  # length 3
).to(device)

# LPIPS (AlexNet by default; 'vgg' also common)
lpips_metric = lpips.LPIPS(net='alex').to(device).eval()


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: c:\Users\devgo\OneDrive\Desktop\fid\.venv\Lib\site-packages\lpips\weights\v0.1\alex.pth


In [13]:
def list_real_fake_files(root):
    real, fake = [], []
    for cls in os.listdir(root):
        cpath = os.path.join(root, cls)
        if not os.path.isdir(cpath):
            continue
        files = [f for f in os.listdir(cpath) if f.lower().endswith(('.png','.jpg','.jpeg'))]
        real += [os.path.join(cpath, f) for f in files if f.startswith("ISIC")]
        fake += [os.path.join(cpath, f) for f in files if not f.startswith("ISIC")]
    return real, fake

real_files, fake_files = list_real_fake_files(root_dir)
print(f"Found {len(real_files)} real and {len(fake_files)} fake images (pooled).")


Found 10015 real and 36920 fake images (pooled).


In [14]:
def compute_overall_ms_ssim_lpips(real_files, fake_files, max_pairs=1000, shuffle=True):
    # Pair by random matching across the pooled sets
    n = min(len(real_files), len(fake_files), max_pairs)
    if shuffle:
        random.shuffle(real_files)
        random.shuffle(fake_files)
    real_files = real_files[:n]
    fake_files = fake_files[:n]

    ms_scores, lp_scores = [], []

    for r_path, f_path in tqdm(zip(real_files, fake_files), total=n):
        try:
            r_img = Image.open(r_path).convert("RGB")
            f_img = Image.open(f_path).convert("RGB")
        except Exception as e:
            # Skip unreadable images
            print("Read error:", e)
            continue

        # Prepare tensors
        r_t = to_01(r_img).unsqueeze(0).to(device)         # [0,1]
        f_t = to_01(f_img).unsqueeze(0).to(device)         # [0,1]

        r_t = r_t.clamp(0, 1)
        f_t = f_t.clamp(0, 1)

        # MS-SSIM (expects [0,1])
        try:
            ms_val = ms_ssim_metric(r_t, f_t).item()
            if not np.isnan(ms_val):
                ms_scores.append(ms_val)
        except Exception as e:
            print("MS-SSIM error:", e)

        # LPIPS (expects [-1,1])
        try:
            r_m11 = to_m11(r_t)
            f_m11 = to_m11(f_t)
            lp_val = lpips_metric(r_m11, f_m11).item()
            if not np.isnan(lp_val):
                lp_scores.append(lp_val)
        except Exception as e:
            print("LPIPS error:", e)

    ms_mean = float(np.nanmean(ms_scores)) if len(ms_scores) else float('nan')
    lp_mean = float(np.nanmean(lp_scores)) if len(lp_scores) else float('nan')
    return ms_mean, lp_mean, ms_scores, lp_scores

overall_ms, overall_lp, ms_list, lp_list = compute_overall_ms_ssim_lpips(
    real_files, fake_files, max_pairs=1000, shuffle=True
)

print(f"\n✅ Overall MS-SSIM: {overall_ms:.4f}")
print(f"✅ Overall LPIPS:   {overall_lp:.4f}")
print(f"(Pairs used: {len(ms_list)} for MS-SSIM, {len(lp_list)} for LPIPS)")


100%|██████████| 1000/1000 [00:08<00:00, 122.22it/s]


✅ Overall MS-SSIM: 0.5635
✅ Overall LPIPS:   0.4395
(Pairs used: 1000 for MS-SSIM, 1000 for LPIPS)


In [15]:
# Check for NaNs/empties
assert not (np.isnan(overall_ms) and len(ms_list) > 0), "MS-SSIM produced NaN unexpectedly."
assert not (np.isnan(overall_lp) and len(lp_list) > 0), "LPIPS produced NaN unexpectedly."

# Basic distribution glimpse
print("MS-SSIM (min/median/max):",
      np.min(ms_list) if ms_list else None,
      np.median(ms_list) if ms_list else None,
      np.max(ms_list) if ms_list else None)

print("LPIPS (min/median/max):",
      np.min(lp_list) if lp_list else None,
      np.median(lp_list) if lp_list else None,
      np.max(lp_list) if lp_list else None)


MS-SSIM (min/median/max): 0.0 0.5746680200099945 0.8174684643745422
LPIPS (min/median/max): 0.19865837693214417 0.4373234659433365 0.7266722321510315


In [17]:
def compute_overall_lpips(real_files, fake_files, random_pairing=True, max_pairs=1000):
	"""
	Compute LPIPS between real and fake images using either random or class-matched pairing
	"""
	# Group files by class
	real_by_class = {}
	fake_by_class = {}
	
	for f in real_files:
		cls = f.split(os.sep)[-2]  # Get class from path
		if cls not in real_by_class:
			real_by_class[cls] = []
		real_by_class[cls].append(f)
		
	for f in fake_files:
		cls = f.split(os.sep)[-2]
		if cls not in fake_by_class:
			fake_by_class[cls] = []
		fake_by_class[cls].append(f)
	
	pairs = []
	if random_pairing:
		# Random pairing across all classes
		real_all = real_files.copy()
		fake_all = fake_files.copy()
		random.shuffle(real_all)
		random.shuffle(fake_all)
		pairs = list(zip(real_all[:max_pairs], fake_all[:max_pairs]))
	else:
		# Class-matched pairing
		for cls in real_by_class:
			if cls in fake_by_class:
				real_cls = real_by_class[cls].copy()
				fake_cls = fake_by_class[cls].copy()
				random.shuffle(real_cls)
				random.shuffle(fake_cls)
				n = min(len(real_cls), len(fake_cls))
				pairs.extend(list(zip(real_cls[:n], fake_cls[:n])))
		# Limit total pairs
		random.shuffle(pairs)
		pairs = pairs[:max_pairs]
	
	# Compute LPIPS for pairs
	lpips_scores = []
	for real_path, fake_path in tqdm(pairs, desc="Computing LPIPS"):
		try:
			real_img = Image.open(real_path).convert("RGB")
			fake_img = Image.open(fake_path).convert("RGB")
			
			real_t = to_01(real_img).unsqueeze(0).to(device)
			fake_t = to_01(fake_img).unsqueeze(0).to(device)
			
			# LPIPS expects [-1,1] range
			real_t = real_t * 2.0 - 1.0
			fake_t = fake_t * 2.0 - 1.0
			
			lpips_val = lpips_metric(real_t, fake_t).item()
			if not np.isnan(lpips_val):
				lpips_scores.append(lpips_val)
		except Exception as e:
			print(f"Error processing {real_path} or {fake_path}: {e}")
			
	return np.mean(lpips_scores) if lpips_scores else float('nan')

# Compute LPIPS using both strategies
overall_random_lpips = compute_overall_lpips(real_files, fake_files, random_pairing=True)
overall_class_matched_lpips = compute_overall_lpips(real_files, fake_files, random_pairing=False)

print("Random pooled LPIPS:", overall_random_lpips)
print("Class-matched pooled LPIPS:", overall_class_matched_lpips)


Computing LPIPS: 100%|██████████| 1000/1000 [00:04<00:00, 205.85it/s]

Random pooled LPIPS: 0.44396381808817387
Class-matched pooled LPIPS: 0.41377529214322567


In [19]:
# Assumes: to_01, to_m11, lpips_model, device, root_dir already set up

import os, numpy as np, pandas as pd
from PIL import Image
from tqdm import tqdm

def compute_class_matched_pairs(root_dir, max_per_class=500):
    rows = []
    for cls in sorted(os.listdir(root_dir)):
        cpath = os.path.join(root_dir, cls)
        if not os.path.isdir(cpath): continue
        reals = sorted([f for f in os.listdir(cpath) if f.startswith("ISIC")])
        fakes = sorted([f for f in os.listdir(cpath) if not f.startswith("ISIC")])
        if not reals or not fakes: continue
        n = min(len(reals), len(fakes), max_per_class)
        for i in range(n):
            r = Image.open(os.path.join(cpath, reals[i % len(reals)])).convert("RGB")
            f = Image.open(os.path.join(cpath, fakes[i % len(fakes)])).convert("RGB")
            r_t = to_01(r).unsqueeze(0).to(device)
            f_t = to_01(f).unsqueeze(0).to(device)
            val = lpips_model(to_m11(r_t), to_m11(f_t)).item()
            rows.append((cls, os.path.join(cpath, reals[i % len(reals)]),
                         os.path.join(cpath, fakes[i % len(fakes)]), val))
    return pd.DataFrame(rows, columns=["class","real","fake","lpips"])

df_pairs = compute_class_matched_pairs(root_dir, max_per_class=300)

# Per-class means, counts
per_class = df_pairs.groupby("class")["lpips"].agg(["mean","median","count"]).sort_values("mean", ascending=False)
print(per_class)

# Top outliers (highest LPIPS)
print("\nTop 30 worst pairs:")
print(df_pairs.sort_values("lpips", ascending=False).head(30))


           mean    median  count
class                           
BKL    0.425631  0.407772    300
MEL    0.422315  0.421349    300
BCC    0.398932  0.394864    300
VASC   0.374240  0.373811    142
AKIEC  0.364673  0.354618    300
DF     0.362841  0.362499    115

Top 30 worst pairs:
      class                                               real  \
875     BKL  C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data...   
783     BKL  C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data...   
707     BKL  C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data...   
603     BKL  C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data...   
1240    MEL  C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data...   
706     BKL  C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data...   
308     BCC  C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data...   
1391   VASC  C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data...   
1118    MEL  C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data...   
792     BKL  C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\

In [26]:
import os
import torch
from PIL import Image
from torchvision import transforms
import lpips
from pytorch_msssim import ms_ssim

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Preprocessing (resize all to same size, normalize)
transform = transforms.Compose([
    transforms.Resize((256, 256)),   # match size to avoid errors
    transforms.ToTensor()
])


In [27]:
def compute_ms_ssim_lpips(real_dir, fake_dir, max_images=100):
    """Compute MS-SSIM and LPIPS between real and fake images in given dirs"""
    real_files = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.startswith("ISIC")]
    fake_files = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if not f.startswith("ISIC")]

    # Balance number of images
    num_images = min(len(real_files), len(fake_files), max_images)
    real_files, fake_files = real_files[:num_images], fake_files[:num_images]

    if num_images == 0:
        return None, None

    # Load LPIPS model
    lpips_model = lpips.LPIPS(net='alex').to(device)

    ms_ssim_scores, lpips_scores = [], []
    for r, f in zip(real_files, fake_files):
        real_img = transform(Image.open(r).convert("RGB")).unsqueeze(0).to(device)
        fake_img = transform(Image.open(f).convert("RGB")).unsqueeze(0).to(device)

        # MS-SSIM
        ms_val = ms_ssim(real_img, fake_img, data_range=1.0, size_average=True).item()
        ms_ssim_scores.append(ms_val)

        # LPIPS
        lp_val = lpips_model(real_img, fake_img).item()
        lpips_scores.append(lp_val)

    return sum(ms_ssim_scores)/len(ms_ssim_scores), sum(lpips_scores)/len(lpips_scores)
